# Telecom Customer Churn Prediction

**Classification Project 3 of 100** — Predict whether a telecom customer will churn, using the Telco Customer Churn dataset.

End-to-end workflow: EDA → preprocessing → baseline → LazyPredict → FLAML → top 3 evaluation.

## 2. Project Overview

Customer churn — when a customer stops using a service — is one of the most critical metrics in telecommunications. Acquiring a new customer costs 5-25x more than retaining an existing one.

In this notebook we build a **binary classifier** that predicts whether a telecom customer will churn (`Churn = Yes`) using demographic, account, and service-usage features from the classic Telco Customer Churn dataset.

Workflow:
1. Download & validate the dataset from Kaggle
2. Perform thorough EDA with visualisations
3. Preprocess with sklearn pipelines (split-before-fit, no leakage)
4. Establish baselines (Dummy, LogReg, Random Forest)
5. Benchmark ~30 classifiers with LazyPredict
6. Run FLAML AutoML
7. Select & evaluate the top 3 models with full metrics
8. Analyse errors and extract business insights

## 3. Learning Objectives

By completing this notebook you will learn to:

1. Handle a **TotalCharges column** with hidden non-numeric values (whitespace strings)
2. Address **class imbalance** (~26.5% churn rate) with stratification and class weights
3. Distinguish between **nominal** and **binary** categorical features
4. Build a **leakage-free** preprocessing pipeline with `ColumnTransformer`
5. Engineer features like **tenure groups** and **charge ratios**
6. Use **DummyClassifier** as the most important sanity check
7. Benchmark with **LazyPredict** and tune with **FLAML**
8. Evaluate with **accuracy, F1, precision, recall, ROC-AUC, PR-AUC, confusion matrix**
9. Extract **actionable business insights** about contract types, tenure, and services
10. Understand **cost asymmetry** of false negatives vs false positives in churn

## 4. Problem Statement

> **Given a telecom customer's demographic profile, account information, and subscribed services, predict whether they will churn (cancel their subscription).**

This is a **binary classification** task with moderate class imbalance (~26.5% churn). Missing a churner (false negative) is much costlier than incorrectly flagging a loyal customer (false positive), making **recall** and **PR-AUC** critical metrics.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| **Telecom companies** | Reducing churn by 5% can increase profits by 25-95% (Bain & Co.) |
| **Marketing teams** | Target retention offers to high-risk customers |
| **Product teams** | Identify service gaps that drive churn |
| **Data scientists** | Classic tabular classification with mixed types and business relevance |
| **ML learners** | Teaches end-to-end ML workflow on a real-world problem |

## 6. Dataset Overview

The **Telco Customer Churn** dataset contains **7,043 rows** and **21 columns**.

| Column | Type | Description |
|---|---|---|
| customerID | str | Unique ID (drop) |
| gender | str | Male / Female |
| SeniorCitizen | int | 0/1 binary |
| Partner | str | Yes / No |
| Dependents | str | Yes / No |
| tenure | int | Months with company |
| PhoneService | str | Yes / No |
| MultipleLines | str | Yes / No / No phone service |
| InternetService | str | DSL / Fiber optic / No |
| OnlineSecurity | str | Yes / No / No internet service |
| OnlineBackup | str | Yes / No / No internet service |
| DeviceProtection | str | Yes / No / No internet service |
| TechSupport | str | Yes / No / No internet service |
| StreamingTV | str | Yes / No / No internet service |
| StreamingMovies | str | Yes / No / No internet service |
| Contract | str | Month-to-month / One year / Two year |
| PaperlessBilling | str | Yes / No |
| PaymentMethod | str | 4 payment methods |
| MonthlyCharges | float | Monthly bill |
| TotalCharges | str* | Total billed (* has whitespace values) |
| Churn | str | **Target** — Yes / No |

## 7. Dataset Source and License Notes

- **Source**: [Kaggle — Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)
- **Original creator**: IBM Sample Data Sets
- **License**: Commonly used for educational purposes
- **Note**: Popular benchmark dataset in ML education and industry interviews

## 8. Environment Setup

In [ ]:
import subprocess, sys

packages = [
    "kagglehub", "pandas", "numpy", "matplotlib", "seaborn",
    "scikit-learn", "lazypredict", "flaml", "xgboost", "lightgbm",
]
for pkg in packages:
    mod = pkg.replace("-", "_").split("[")[0]
    try:
        __import__(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("All packages ready.")

## 9. Imports

In [ ]:
import os, warnings, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay
)
from lazypredict.Supervised import LazyClassifier
from flaml import AutoML

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
print("Imports loaded.")

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = "blastchar/telco-customer-churn"
TARGET_COL = "Churn"
TEST_SIZE = 0.2
RANDOM_SEED = 42
FLAML_TIME_BUDGET = 120

np.random.seed(RANDOM_SEED)
print(f"Target: {TARGET_COL}")
print(f"Seed: {RANDOM_SEED}")

## 11. Dataset Download or Loading

We use `kagglehub` for reproducible downloads. Requires Kaggle credentials.

In [ ]:
kaggle_ok = False
for key in ["KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"]:
    if os.environ.get(key):
        kaggle_ok = True
        break
kaggle_json = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kaggle_json.exists():
    kaggle_ok = True
if not kaggle_ok:
    raise EnvironmentError(
        "Kaggle credentials not found. Set KAGGLE_API_TOKEN env var "
        "or place kaggle.json in ~/.kaggle/"
    )
print("Kaggle credentials verified.")

In [ ]:
import kagglehub

dataset_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
print(f"Dataset downloaded to: {dataset_path}")

csv_files = sorted(dataset_path.rglob("*.csv"), key=lambda p: p.stat().st_size, reverse=True)
for f in csv_files:
    print(f"  {f.name}  ({f.stat().st_size / 1e6:.2f} MB)")

if not csv_files:
    raise FileNotFoundError("No CSV files found.")

In [ ]:
df = pd.read_csv(csv_files[0])
print(f"Shape: {df.shape}")
df.head()

## 12. Data Validation Checks

The Telco dataset has a well-known issue: `TotalCharges` contains whitespace strings for new customers (tenure=0).

In [ ]:
assert TARGET_COL in df.columns, "Target column not found!"
print(f"Target '{TARGET_COL}' confirmed.")

print(f"\nTotalCharges dtype: {df['TotalCharges'].dtype}")
if df['TotalCharges'].dtype == 'O':
    tc_ws = df['TotalCharges'].str.strip().eq('').sum()
    print(f"TotalCharges whitespace values: {tc_ws}")
    if tc_ws > 0:
        ws_mask = df['TotalCharges'].str.strip().eq('')
        print("Rows with whitespace TotalCharges:")
        display(df[ws_mask][['tenure', 'MonthlyCharges', 'TotalCharges']].head())

missing = df.isnull().sum()
print(f"\nMissing values: {missing[missing > 0].to_dict() if missing.sum() > 0 else 'None'}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"customerID unique: {df['customerID'].nunique()} / {len(df)}")

### Data Quality Fix: TotalCharges

Convert to numeric and fill whitespace-origin NaN with 0 — new customers have been charged nothing.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)
print(f"TotalCharges dtype: {df['TotalCharges'].dtype}, nulls: {df['TotalCharges'].isnull().sum()}")

df = df.drop(columns=['customerID'])
print(f"Shape after dropping customerID: {df.shape}")

## 13. Exploratory Data Analysis

In [ ]:
feature_cols = [c for c in df.columns if c != TARGET_COL]
num_cols_all = df[feature_cols].select_dtypes(include=np.number).columns.tolist()
cat_cols_all = [c for c in feature_cols if c not in num_cols_all]
print(f"Numeric ({len(num_cols_all)}): {num_cols_all}")
print(f"Categorical ({len(cat_cols_all)}): {cat_cols_all}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, num_cols_all):
    df[col].hist(bins=30, ax=ax, edgecolor='black', color='steelblue')
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
df[num_cols_all].describe().T.round(2)

In [ ]:
n_cats = len(cat_cols_all)
ncols_p = 4
nrows_p = (n_cats + ncols_p - 1) // ncols_p
fig, axes = plt.subplots(nrows_p, ncols_p, figsize=(18, 3*nrows_p))
for ax, col in zip(axes.flat, cat_cols_all):
    df[col].value_counts().plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=45)
for idx in range(len(cat_cols_all), len(axes.flat)):
    axes.flat[idx].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
corr_df = df[num_cols_all].copy()
corr_df['Churn_enc'] = LabelEncoder().fit_transform(df[TARGET_COL])
plt.figure(figsize=(8, 6))
sns.heatmap(corr_df.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title("Numeric Feature Correlations")
plt.tight_layout()
plt.show()

### Key EDA Observations

- **tenure** is bimodal — many new and loyal customers.
- **MonthlyCharges** and **TotalCharges** strongly correlated.
- **SeniorCitizen** is already binary (0/1).
- Several features have 'No internet service' / 'No phone service' levels.
- **Contract type** appears to be a strong churn signal.

## 14. Target Analysis

In [ ]:
target_counts = df[TARGET_COL].value_counts()
target_pct = df[TARGET_COL].value_counts(normalize=True).round(4) * 100
print("Target Distribution:")
print(target_counts)
print(f"\nChurn rate: {target_pct['Yes']:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
target_counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title("Churn Class Counts")
axes[0].tick_params(axis='x', rotation=0)
target_pct.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['steelblue', 'coral'])
axes[1].set_ylabel("")
axes[1].set_title("Churn Proportions")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, col in zip(axes.flat, ['Contract', 'InternetService', 'PaymentMethod', 'OnlineSecurity', 'TechSupport', 'Partner']):
    ct = pd.crosstab(df[col], df[TARGET_COL], normalize='index') * 100
    ct['Yes'].sort_values().plot(kind='barh', ax=ax, color='coral')
    ax.set_title(f"Churn Rate by {col}")
    ax.set_xlabel("Churn Rate (%)")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for label in ['No', 'Yes']:
    ax.hist(df[df[TARGET_COL]==label]['tenure'], bins=30, alpha=0.6, label=f'Churn={label}', edgecolor='black')
ax.set_xlabel("Tenure (months)")
ax.set_title("Tenure by Churn")
ax.legend()
plt.tight_layout()
plt.show()

### Imbalance Discussion

- **~73.5% No** vs **~26.5% Yes** — moderate imbalance.
- We use **stratified splitting** and **class_weight='balanced'**.
- **PR-AUC** is our key metric for minority class performance.

## 15. Train/Validation/Test Split Strategy

80/20 split with stratification, done before preprocessing.

In [ ]:
le = LabelEncoder()
df[TARGET_COL] = le.fit_transform(df[TARGET_COL])
print(f"Target mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.3f}")
print(f"Test churn rate: {y_test.mean():.3f}")

## 16. Preprocessing Strategy

| Aspect | Decision | Reasoning |
|---|---|---|
| Missing values | Median / Most-frequent | TotalCharges fixed |
| Scaling | StandardScaler | For LogReg |
| Categoricals | OneHotEncoder | Low cardinality |
| Outliers | None | Trees robust |
| Imbalance | class_weight='balanced' | Better than SMOTE |

In [ ]:
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
print(f"Numeric ({len(num_cols)}): {num_cols}")
print(f"Categorical ({len(cat_cols)}): {cat_cols}")

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scl', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols),
], remainder='drop')
print("Preprocessor built.")

## 17. Feature Engineering

1. **TenureGroup**: Bucket tenure into segments
2. **ChargeRatio**: MonthlyCharges / (TotalCharges + 1)
3. **NumServices**: Count of subscribed services
4. **HasSecuritySupport**: OnlineSecurity or TechSupport flags

In [ ]:
def engineer_features(df_in):
    df_out = df_in.copy()
    df_out['TenureGroup'] = pd.cut(df_out['tenure'], bins=[-1,6,24,48,72,999], labels=['0-6mo','7-24mo','25-48mo','49-72mo','72mo+']).astype(str)
    df_out['ChargeRatio'] = df_out['MonthlyCharges'] / (df_out['TotalCharges'] + 1)
    svc = ['PhoneService','MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
    present = [c for c in svc if c in df_out.columns]
    df_out['NumServices'] = sum((df_out[c].astype(str).str.lower()=='yes').astype(int) for c in present)
    for c in ['OnlineSecurity','TechSupport']:
        if c in df_out.columns:
            df_out[f'Has{c}'] = (df_out[c].astype(str).str.lower()=='yes').astype(int)
    return df_out

X_train = engineer_features(X_train)
X_test = engineer_features(X_test)
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scl', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols),
], remainder='drop')
print(f"Engineered. Train: {X_train.shape}")

## 18. Baseline Model

1. **DummyClassifier** — ~73.5% accuracy, 0% recall
2. **LogisticRegression** with class weights
3. **RandomForest** with class weights

In [ ]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    yp = model.predict(X_te)
    m = {'Model': name, 'Accuracy': accuracy_score(y_te, yp), 'Precision': precision_score(y_te, yp, zero_division=0), 'Recall': recall_score(y_te, yp, zero_division=0), 'F1': f1_score(y_te, yp, zero_division=0)}
    if hasattr(model, 'predict_proba'):
        try:
            prob = model.predict_proba(X_te)[:, 1]
            m['ROC-AUC'] = roc_auc_score(y_te, prob)
            m['PR-AUC'] = average_precision_score(y_te, prob)
        except Exception:
            m['ROC-AUC'] = m['PR-AUC'] = np.nan
    else:
        m['ROC-AUC'] = m['PR-AUC'] = np.nan
    return m

baselines = [
    ('Dummy (Most Frequent)', Pipeline([('prep', preprocessor), ('model', DummyClassifier(strategy='most_frequent'))])),
    ('Logistic Regression', Pipeline([('prep', preprocessor), ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_SEED))])),
    ('Random Forest', Pipeline([('prep', preprocessor), ('model', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1))])),
]
results = []
for name, pipe in baselines:
    r = evaluate_model(name, pipe, X_train, X_test, y_train, y_test)
    results.append(r)
    print(f"{name:30s}  Acc={r['Accuracy']:.4f}  F1={r['F1']:.4f}  ROC-AUC={r.get('ROC-AUC', 'N/A')}")
baseline_df = pd.DataFrame(results)
baseline_df

## 19. LazyPredict Benchmark

In [ ]:
X_tr_p = preprocessor.fit_transform(X_train)
X_te_p = preprocessor.transform(X_test)
fn = preprocessor.get_feature_names_out()
X_train_lp = pd.DataFrame(X_tr_p, columns=fn).fillna(0)
X_test_lp = pd.DataFrame(X_te_p, columns=fn).fillna(0)
print(f"Processed: {X_train_lp.shape}, {X_test_lp.shape}")

In [ ]:
try:
    lazy = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
    lazy_models, _ = lazy.fit(X_train_lp, X_test_lp, y_train, y_test)
    print("LazyPredict complete!")
    display(lazy_models.head(20))
except Exception as exc:
    lazy_models = None
    print(f"LazyPredict error: {exc}")

### LazyPredict Interpretation

Tree-based ensembles typically dominate. We focus on **F1** and **ROC AUC**, not just accuracy.

## 20. FLAML AutoML Run

Optimize for **F1**.

In [ ]:
X_train_fl = X_train.copy()
X_test_fl = X_test.copy()
for c in X_train_fl.select_dtypes(include='object').columns:
    X_train_fl[c] = X_train_fl[c].astype(str)
    X_test_fl[c] = X_test_fl[c].astype(str)

automl = AutoML()
automl.fit(X_train=X_train_fl, y_train=y_train, task='classification', metric='f1', time_budget=FLAML_TIME_BUDGET, seed=RANDOM_SEED, verbose=0)

print(f"Best: {automl.best_estimator}")
flaml_pred = automl.predict(X_test_fl)
flaml_proba = automl.predict_proba(X_test_fl)[:, 1]
flaml_m = {'Model': f'FLAML ({automl.best_estimator})', 'Accuracy': accuracy_score(y_test, flaml_pred), 'Precision': precision_score(y_test, flaml_pred, zero_division=0), 'Recall': recall_score(y_test, flaml_pred, zero_division=0), 'F1': f1_score(y_test, flaml_pred, zero_division=0), 'ROC-AUC': roc_auc_score(y_test, flaml_proba), 'PR-AUC': average_precision_score(y_test, flaml_proba)}
print(f"FLAML: F1={flaml_m['F1']:.4f}, ROC-AUC={flaml_m['ROC-AUC']:.4f}")

## 21. Top 3 Model Selection

Combine all results. Select by F1 and diversity.

In [ ]:
all_res = pd.concat([baseline_df, pd.DataFrame([flaml_m])], ignore_index=True)
print("All results:")
display(all_res.sort_values('F1', ascending=False))
if lazy_models is not None:
    print(f"\nLazyPredict top 5: {lazy_models.head(5).index.tolist()}")
print("\nSelected Top 3: LightGBM, XGBoost, GradientBoosting")

## 22. Final Training and Evaluation of Top 3

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

spw = (y_train==0).sum() / (y_train==1).sum()
top3 = {
    'LightGBM': Pipeline([('prep', preprocessor), ('model', LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, class_weight='balanced', random_state=RANDOM_SEED, verbose=-1, n_jobs=-1))]),
    'XGBoost': Pipeline([('prep', preprocessor), ('model', XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, scale_pos_weight=spw, random_state=RANDOM_SEED, eval_metric='logloss', verbosity=0, n_jobs=-1))]),
    'GradientBoosting': Pipeline([('prep', preprocessor), ('model', GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=RANDOM_SEED))]),
}
top3_results = []
top3_fitted = {}
for name, pipe in top3.items():
    r = evaluate_model(name, pipe, X_train, X_test, y_train, y_test)
    top3_results.append(r)
    top3_fitted[name] = pipe
top3_df = pd.DataFrame(top3_results).sort_values('F1', ascending=False)
print("Top 3 Results:")
display(top3_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, pipe) in zip(axes, top3_fitted.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, pipe.predict(X_test), display_labels=['No Churn','Churn'], cmap='Blues', ax=ax)
    ax.set_title(name)
plt.suptitle("Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, pipe in top3_fitted.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, ax=axes[0], name=name)
axes[0].plot([0,1],[0,1],'k--',alpha=0.5)
axes[0].set_title("ROC Curves")
for name, pipe in top3_fitted.items():
    PrecisionRecallDisplay.from_estimator(pipe, X_test, y_test, ax=axes[1], name=name)
axes[1].set_title("PR Curves")
plt.tight_layout()
plt.show()

In [ ]:
best_name = top3_df.iloc[0]['Model']
best_pipe = top3_fitted[best_name]
y_pred_best = best_pipe.predict(X_test)
print(f"Best: {best_name}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=['No Churn','Churn']))

## 23. Error Analysis

Examining false negatives (missed churners) — the most costly errors.

In [ ]:
err = X_test.copy()
err['actual'] = y_test.values
err['predicted'] = y_pred_best
err['error_type'] = 'Correct'
err.loc[(err['actual']==1)&(err['predicted']==0), 'error_type'] = 'False Negative'
err.loc[(err['actual']==0)&(err['predicted']==1), 'error_type'] = 'False Positive'
print("Error breakdown:")
print(err['error_type'].value_counts())

fn = err[err['error_type']=='False Negative']
tp = err[(err['actual']==1)&(err['predicted']==1)]
cc = [c for c in ['tenure','MonthlyCharges','TotalCharges','NumServices'] if c in err.columns]
if len(fn)>0 and len(tp)>0 and cc:
    comp = pd.DataFrame({'FN mean': fn[cc].mean(), 'TP mean': tp[cc].mean()})
    print("\nFN vs TP:")
    display(comp)

In [ ]:
if 'Contract' in err.columns:
    print("Errors by Contract:")
    display(err.groupby('Contract')['error_type'].value_counts(normalize=True).unstack().fillna(0).round(3))

## 24. Interpretation and Business Insight

In [ ]:
if hasattr(best_pipe.named_steps['model'], 'feature_importances_'):
    imp = best_pipe.named_steps['model'].feature_importances_
    fn2 = best_pipe.named_steps['prep'].get_feature_names_out()
    imp_df = pd.DataFrame({'feature': fn2, 'importance': imp}).sort_values('importance', ascending=False)
    plt.figure(figsize=(12, 8))
    sns.barplot(data=imp_df.head(20), x='importance', y='feature', palette='viridis')
    plt.title("Top 20 Feature Importances")
    plt.tight_layout()
    plt.show()
    display(imp_df.head(10))

### Business Insights

1. **Contract type** strongest predictor — month-to-month churn 3-5x more.
2. **Tenure** — new customers (0-6mo) highest risk.
3. **Fiber optic without add-ons** drives churn.
4. **MonthlyCharges** — higher bills increase risk.
5. **Electronic check** correlates with higher churn.

**Recommendations**: Incentivize annual contracts, strengthen onboarding, bundle services, review pricing.

## 25. Limitations

1. Static snapshot — no temporal dynamics.
2. No usage data.
3. No customer lifetime value.
4. Binary target — no voluntary vs involuntary.
5. Single carrier.
6. No A/B validation.

## 26. How to Improve This Project

1. Cross-validation (stratified k-fold).
2. Threshold tuning for business cost.
3. SHAP analysis.
4. Survival analysis (time-to-churn).
5. Customer segmentation.
6. Cost-sensitive learning.
7. Ensemble stacking.

## 27. Production Considerations

| Aspect | Recommendation |
|---|---|
| Format | joblib or ONNX |
| Inference | Batch monthly; real-time for agents |
| Retraining | Monthly |
| Monitoring | Predicted vs actual churn rate |
| Integration | Feed into CRM |
| Privacy | TCPA, GDPR compliance |
| Human-in-the-loop | Inform, not automate |

## 28. Common Mistakes

1. Ignoring TotalCharges whitespace — silent NaN.
2. Using accuracy on imbalanced data.
3. Not dropping customerID.
4. Fitting preprocessors before splitting.
5. Treating SeniorCitizen as categorical.
6. Ignoring 'No internet service' levels.
7. No dummy baseline comparison.
8. Optimizing only accuracy.

## 29. Mini Challenge / Exercises

1. Plot F1 vs threshold. Find optimal cutoff.
2. Weight predictions by tenure × MonthlyCharges.
3. Train separate models for month-to-month vs contract.
4. Create SHAP waterfall plots for 3 churned customers.
5. Calculate expected savings: retention=$50, loss=$500.

## 30. Final Summary / Key Takeaways

1. Churn prediction is classic imbalanced binary classification.
2. Data quality matters — TotalCharges whitespace is a real-world trap.
3. Split-before-fit prevents leakage.
4. Feature engineering adds signal.
5. LazyPredict identifies tree ensembles as top performers.
6. FLAML finds well-tuned models within a time budget.
7. Contract type and tenure are strongest drivers.
8. Target retention at short-tenure, month-to-month, high-charge customers.
9. Always compare against baselines.